# 03 — Extract Regular Lines

Filter raw vehicle position data (`vehicle_positions_2025_07_22.csv`) to retain only the **46 regular DVB lines** used in the analysis.

**Steps:**
1. Define the target regular line numbers
2. Load raw CSV with Polars (lazy scan for memory efficiency)
3. Filter to regular lines
4. Inspect coverage and basic stats
5. Save filtered data to `data/regular_lines_2025_07_22.parquet`

## 0. Imports and paths

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import date
from pathlib import Path
import polars as pl

In [ ]:
DATA_DIR = Path("..") / "data"
RAW_CSV  = DATA_DIR / "vehicle_positions_2025_07_22.csv"

## 1. Define regular lines

In [4]:
REGULAR_LINES = [
    1, 2, 3, 4, 6, 7, 8, 9,
    10, 11, 12, 13,
    47,
    59, 60, 61, 62, 63, 64, 65, 66, 68,
    70, 72, 73, 74, 77, 78, 79,
    80, 81, 84, 85, 86, 87, 88,
    104, 106, 108, 112, 113,
    162, 166,
    477, 520, 521,
]

print(f"{len(REGULAR_LINES)} regular lines: {REGULAR_LINES}")

46 regular lines: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 47, 59, 60, 61, 62, 63, 64, 65, 66, 68, 70, 72, 73, 74, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 104, 106, 108, 112, 113, 162, 166, 477, 520, 521]


## 2. Load and filter

`ort_nr_start_vvo` and `ort_nr_ziel_vvo` contain mixed values (e.g. `MBR379191`) so we override their schema to `Utf8`.

In [5]:
df = (
    pl.scan_csv(
        RAW_CSV,
        infer_schema_length=10_000,
        schema_overrides={"ort_nr_start_vvo": pl.Utf8, "ort_nr_ziel_vvo": pl.Utf8},
    )
    .filter(pl.col("linie").is_in(REGULAR_LINES))
    .collect()
)

print(f"Rows after filter: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

Rows after filter: 24,261,675  |  Columns: 25


In [8]:
df_week = df.filter(
    pl.col("tst").str.slice(0, 10).is_between(pl.lit("2025-07-28"), pl.lit("2025-08-03"))
)

print(f"Rows 2025-07-28 – 2025-08-03 (Dresden local time): {df_week.shape[0]:,}")

Rows 2025-07-28 – 2025-08-03 (Dresden local time): 10,708,589


In [9]:
date_col = pl.col("tst").str.slice(0, 10)
out_of_range = df_week.filter(
    date_col.lt(pl.lit("2025-07-28")) | date_col.gt(pl.lit("2025-08-03"))
)

print(f"Out-of-range rows: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range.select("tst").head(10))

Out-of-range rows: 0


## 4. Split tram / bus

In [11]:
TRAM_LINES = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 20]

df_tram = df_week.filter(pl.col("linie").is_in(TRAM_LINES))
df_bus  = df_week.filter(~pl.col("linie").is_in(TRAM_LINES))

print(f"Tram rows: {df_tram.shape[0]:,}  |  lines: {sorted(df_tram['linie'].unique().to_list())}")
print(f"Bus  rows: {df_bus.shape[0]:,}  |  lines: {sorted(df_bus['linie'].unique().to_list())}")

Tram rows: 4,980,084  |  lines: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13]
Bus  rows: 5,728,505  |  lines: [47, 59, 60, 61, 62, 63, 64, 65, 66, 68, 70, 72, 73, 74, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 104, 106, 108, 112, 113, 162, 166, 477, 520, 521]


In [13]:
OUT_TRAM = DATA_DIR / "tram_2025-07-28_2025-08-03.parquet"
OUT_BUS  = DATA_DIR / "bus_2025-07-28_2025-08-03.parquet"

df_tram.write_parquet(OUT_TRAM)
df_bus.write_parquet(OUT_BUS)
print(f"Tram → {OUT_TRAM}  ({OUT_TRAM.stat().st_size / 1e6:.1f} MB)")
print(f"Bus  → {OUT_BUS}  ({OUT_BUS.stat().st_size / 1e6:.1f} MB)")

Tram → ../data/tram_2025-07-28_2025-08-03.parquet  (147.7 MB)
Bus  → ../data/bus_2025-07-28_2025-08-03.parquet  (181.2 MB)
